# Phase 3: Baseline — Raw Qwen3-4B with Tools

Run the **un-fine-tuned** Qwen3-4B with tool calling enabled to establish "before" metrics.
This gives us failure examples for the talk and a baseline reward score to compare against SFT and RL.

**Expected failures:** wrong tool selection, malformed JSON, infinite looping, no final output, no reasoning.

## Local MLX Testing (Mac)

Before running the full baseline on Databricks with Unsloth, we can test inference locally using
**mlx-lm** on Apple Silicon. This validates our agent loop and tool calling against a local model.

### Prerequisites (run in terminal)

**1. Install mlx-lm (latest — cache fixes for thinking models are recent) and login to HuggingFace:**
```bash
uv add "mlx-lm[train]" --upgrade
uv run huggingface-cli login
```

**2. Quick CLI test (no server needed):**
```bash
uv run mlx_lm.generate \
  --model mlx-community/Qwen3.5-2B-4bit \
  --prompt "What are Apple's main revenue segments?" \
  --max-tokens 500
```

**3. Start the local server:**
```bash
# Single-request testing
uv run mlx_lm.server \
  --model mlx-community/Qwen3.5-2B-4bit \
  --port 8080

# Batch baseline (concurrent agent runs)
uv run mlx_lm.server \
  --model mlx-community/Qwen3.5-2B-4bit \
  --port 8080 \
  --prompt-concurrency 8 \
  --decode-concurrency 32 \
  --prompt-cache-size 4 \
  --chat-template-args '{"enable_thinking":true}'
```

> **Key flags:**
> - `--prompt-cache-size 4` — caps KV caches in memory (default 10 causes OOM on 16GB Mac).
> - `--prompt-concurrency 8` / `--decode-concurrency 32` — parallel prefill and decode slots.
>
> **Important:** Thinking models (Qwen3.5) have 0% KV cache reuse on the first request after
> server start, causing extremely long reasoning loops. The warm-up cell below sends a throwaway
> request to prime the cache before real work. See [mlx-lm#1042](https://github.com/ml-explore/mlx-lm/pull/1042).

**4. Verify with curl:**
```bash
curl http://localhost:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"messages": [{"role": "user", "content": "Say hello in 5 words"}], "max_tokens": 200}'
```

Once the server is running on port 8080, run the cells below.

In [1]:
# MLX Local Test — requires mlx_lm.server running on port 8080
# Start it first: uv run mlx_lm.server --model mlx-community/Qwen3.5-2B-4bit --port 8080

import json
import sys
from pathlib import Path

from openai import AsyncOpenAI

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.agent import run_tool_calling_agent_chat, TOOL_SCHEMAS_CHAT
from bbb.helpers__data_gen import SYSTEM_PROMPT
from bbb.tools import TOOL_FUNCTIONS
from bbb.helpers__inference import compute_reward

client = AsyncOpenAI(base_url="http://localhost:8080/v1", api_key="none")
VALID_TOOL_NAMES = set(TOOL_FUNCTIONS.keys())

In [2]:
# Warm up the prompt cache — first request after server start produces abnormally
# long thinking due to cold KV cache (0% reuse on thinking models). This throwaway
# call primes the cache so real requests behave normally.
warmup = await client.chat.completions.create(
    model="default_model",
    messages=[{"role": "user", "content": "Say hello."}],
    max_tokens=10,
)

print(f"Cache warmed: {warmup.choices[0].message.reasoning!r}")

Cache warmed: 'Thinking Process:\n\n1.  **Analyze'


Simple test

In [6]:
result = await run_tool_calling_agent_chat(
    client=client,
    model="default_model",
    user_prompt="Briefly, what is AAPL's main revenue segment?",
    system_prompt="You are a helpful assistant. Be concise.",
    max_iterations=5,
    max_tokens=1024,
)

  [1] Reasoning: Thinking Process:  1.  **Analyze the Request:**     *   Subject: Apple Inc. (AAPL)     *   Question: What is their main ...
  [1] Agent finished — produced final response


In [7]:
print(result['output_text'])

<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Subject: Apple Inc. (AAPL)
    *   Question: What is their main revenue segment?
    *   Constraint: Briefly.

2.  **Identify Key Information:**
    *   Apple's core business is software, services, and devices.
    *   The largest single revenue source historically has been hardware (specifically iPhones, Macs, iPads), but now services and software dominate.
    *   However, looking at the latest financial reports (FY2023/2024), the "main" segment is often cited as **Apple Services** or the combination of hardware and software.
    *   Wait, I need to be precise about "main revenue segment." Historically, hardware was the big chunk. But recent data suggests services are growing rapidly and often overtake or exceed hardware in revenue share.
    *   Actually, checking general knowledge: Apple's revenue is split into Hardware, Services, and Digital Content.
    *   Historically, hardware (iPhone/Mac) was the largest segment 

Tool calling

In [8]:
result = await run_tool_calling_agent_chat(
    client=client,
    model="default_model",
    user_prompt="Research AAPL focusing on financial health.",
    system_prompt=SYSTEM_PROMPT,
    tools=TOOL_SCHEMAS_CHAT,
    tool_functions=TOOL_FUNCTIONS,
    max_iterations=5,
    temperature=0.7,
    max_tokens=4096,
)

  [1] Reasoning: The user wants me to research AAPL (Apple) focusing on financial health. I need to gather relevant financial data to pro...
  [1] Called get_financials(ticker='AAPL', statement_type='income', period='annual')
  [1] Called get_financials(ticker='AAPL', statement_type='balance_sheet', period='annual')
  [1] Called get_price_history(ticker='AAPL', period='5y', interval='1mo')
  [1] Called get_recommendations(ticker='AAPL', months_back=12)
  [2] Reasoning: Let me analyze the data I've received:  **Financials (Income Statement):** - Current Period (2025-09-30):    - Total Rev...
  [2] Agent finished — produced final response


In [9]:
# Results
n_tool_calls = sum(len(m.get("tool_calls", [])) for m in result["input"])
print(f"Tool calls: {n_tool_calls}")
print(f"Output length: {len(result['output_text'])} chars")

Tool calls: 4
Output length: 15039 chars


In [10]:
print(result['output_text'])

<think>
Let me analyze the data I've received:

**Financials (Income Statement):**
- Current Period (2025-09-30): 
  - Total Revenue: $416.16 billion
  - Gross Profit: $195.20 billion
  - Operating Income: $133.05 billion
  - Net Income: $112.01 billion (continuing)
  - Diluted EPS: $7.46
  - Net Debt: $62.72 billion (Treasury shares: null, so I'll focus on operating debt)

**Balance Sheet:**
- Net Debt: $62.72 billion (for 2025-09-30)
- Common Shares: 14,773,260,000 (for 2025-09-30)
- Total Equity: $73.73 billion (2025-09-30)
- Working Capital: -$17.67 billion (negative working capital suggests cash considerations)

**Price History (5-year):**
- Min: $121.37, Max: $278.32
- Avg: $187.81
- Total change: +94.31% over 5 years

**Analyst Consensus (12 months):**
- Strong Buy: 6, Buy: 25, Hold: 15, Sell: 1, Strong Sell: 1
- Total Buy/Strong Buy bias is very strong (41 out of 58 analysts)

**Recent Upgrades/Downgrades:**
- Wedbush: Maintained "Outperform" at $350 (no change)
- Morgan Stanle

In [11]:
# Score it
reward = compute_reward(result["input"], VALID_TOOL_NAMES, result.get("reasoning_summaries"))
print(f"\nReward: {reward}")


Reward: {'valid_json': 1.0, 'thinking': 1.0, 'tool_selection': 1.5, 'efficiency': -0.0, 'completion': 1.0, 'no_hallucination': 0.0, 'total': 4.5}


In [12]:
result

{'input': [{'role': 'system',
   'content': "You are a sell-side equity research analyst producing a brief research snapshot.\n\nToday's date is 2026-03-30.\n\n## Instructions\n- Use the available tools to gather the data you need. Be efficient — call only what's necessary.\n- Think briefly about what data points matter most, then call tools.\n- After gathering data, produce a concise equity research snapshot.\n\n## Output Format\nProduce a brief **Equity Research Snapshot** (~half page). Structure:\n\n**{COMPANY} ({TICKER})** | {Sector} | {Market Cap}\n\n**Key Metrics:** Revenue (TTM), EPS, P/E, margins, debt/equity — whatever is most relevant.\n**Recent Developments:** 1-3 bullet points on material news, earnings, or events.\n**Financial Highlights:** Key takeaways from the most recent financials.\n**Price Action:** Current price context, 52-week range, recent trend summary.\n**Analyst Consensus:** Target price, buy/hold/sell split, notable recent upgrades or downgrades.\n**Bull Case

## Batch Async Baseline (MLX)

Fire 10 concurrent agent runs at the local mlx_lm.server to measure throughput and collect aggregate baseline metrics. Each agent makes multiple tool calls, so the real bottleneck is yfinance rate limiting — we use a semaphore to cap concurrent agents.

> **Server concurrency:** mlx_lm.server batches requests natively via `--decode-concurrency`
> (default 32) and `--prompt-concurrency` (default 8). No client-side batching needed for the
> LLM itself — the semaphore below only limits how many agents hit yfinance simultaneously.

In [13]:
import asyncio
import random
import time

from tqdm.asyncio import tqdm_asyncio

from bbb.helpers__data_gen import TICKERS, FOCUS_AREAS, make_user_prompt

# How many agents can run concurrently — limits yfinance pressure, not LLM throughput
CONCURRENCY = 5
N_TICKERS = 5

semaphore = asyncio.Semaphore(CONCURRENCY)
eval_tickers = random.sample(TICKERS, N_TICKERS)
print(f"Evaluating {N_TICKERS} tickers with concurrency={CONCURRENCY}: {eval_tickers}")

Evaluating 5 tickers with concurrency=5: ['PNC', 'DD', 'GD', 'MDB', 'FCX']


In [14]:
async def run_one(ticker: str, focus: str) -> dict:
    """Run a single agent with semaphore-controlled concurrency."""
    async with semaphore:
        try:
            res = await run_tool_calling_agent_chat(
                client=client,
                model="default_model",
                user_prompt=make_user_prompt(ticker, focus),
                system_prompt=SYSTEM_PROMPT,
                tools=TOOL_SCHEMAS_CHAT,
                tool_functions=TOOL_FUNCTIONS,
                max_iterations=5,
                temperature=0.7,
                max_tokens=4096,
                verbose=False,
            )

            n_tool_calls = sum(len(m.get("tool_calls", [])) for m in res["input"])
            reward = compute_reward(res["input"], VALID_TOOL_NAMES, res.get("reasoning_summaries"))

            return {
                "ticker": ticker,
                "focus": focus,
                "n_tool_calls": n_tool_calls,
                "output_len": len(res["output_text"]),
                "reward": reward,
                "output_text": res["output_text"],
                "messages": res["input"],
            }
        except Exception as e:
            return {
                "ticker": ticker,
                "focus": focus,
                "error": str(e),
                "n_tool_calls": 0,
                "output_len": 0,
                "reward": {"total": -3.0},
                "output_text": "",
                "messages": [],
            }

In [15]:
tasks = [run_one(t, random.choice(FOCUS_AREAS)) for t in eval_tickers]

In [16]:
t0 = time.time()
batch_results = await tqdm_asyncio.gather(*tasks, desc="Batch baseline")
elapsed = time.time() - t0

Batch baseline:   0%|          | 0/5 [00:00<?, ?it/s]

Batch baseline: 100%|██████████| 5/5 [03:47<00:00, 45.45s/it]


In [17]:
errors = [r for r in batch_results if "error" in r]
print(f"\nDone: {len(batch_results)} tickers in {elapsed:.1f}s ({elapsed/len(batch_results):.1f}s/ticker)")
print(f"Errors: {len(errors)}")
if errors:
    for e in errors:
        print(f"  {e['ticker']}: {e['error']}")


Done: 5 tickers in 227.3s (45.5s/ticker)
Errors: 0


### Batch Results

In [29]:
# Aggregate metrics
successful = [r for r in batch_results if "error" not in r]
rewards = [r["reward"]["total"] for r in successful]
tool_counts = [r["n_tool_calls"] for r in successful]
output_lens = [r["output_len"] for r in successful]

print(f"=== BATCH BASELINE ({len(successful)}/{len(batch_results)} successful) ===")
print(f"Throughput:   {elapsed:.1f}s total, {elapsed/len(batch_results):.1f}s/ticker")
print(f"Reward:       avg={sum(rewards)/len(rewards):.2f}  min={min(rewards):.2f}  max={max(rewards):.2f}")
print(f"Tool calls:   avg={sum(tool_counts)/len(tool_counts):.1f}  min={min(tool_counts)}  max={max(tool_counts)}")
print(f"Output chars: avg={sum(output_lens)/len(output_lens):.0f}")

components = ["valid_json", "thinking", "tool_selection", "efficiency", "completion", "no_hallucination"]
print("\n--- Per-component averages ---")
for comp in components:
    vals = [r["reward"].get(comp, 0) for r in successful]
    print(f"  {comp:>20}: {sum(vals)/len(vals):+.2f}")

print(f"\n{'Ticker':<8} {'Focus':<45} {'Reward':>7} {'Tools':>6} {'Output':>7}")
print("-" * 80)
for r in sorted(successful, key=lambda x: x["reward"]["total"]):
    rw = r["reward"]
    print(f"{r['ticker']:<8} {r['focus'][:43]:<45} {rw['total']:>+7.2f} {r['n_tool_calls']:>6} {r['output_len']:>7}")

=== BATCH BASELINE (5/5 successful) ===
Throughput:   227.3s total, 45.5s/ticker
Reward:       avg=3.40  min=2.00  max=5.00
Tool calls:   avg=4.0  min=0  max=7
Output chars: avg=3008

--- Per-component averages ---
            valid_json: +0.80
              thinking: +1.00
        tool_selection: +1.40
            efficiency: -0.60
            completion: +0.80
      no_hallucination: +0.00

Ticker   Focus                                          Reward  Tools  Output
--------------------------------------------------------------------------------
DD       growth potential and market expansion oppor     +2.00      0    1383
GD       revenue trends and profitability trajectory     +2.50      7    4194
FCX      recent news and analyst sentiment               +3.00      6       0
PNC      competitive position and market share dynam     +4.50      3    5852
MDB      risk factors and key challenges ahead           +5.00      4    3613


In [30]:
# Save batch results — full trajectories included for training/analysis
output_dir = PROJECT_ROOT / "data" / "bbb"
output_dir.mkdir(parents=True, exist_ok=True)

results_path = output_dir / "baseline_results_mlx_2b.jsonl"
with open(results_path, "w") as f:
    for r in batch_results:
        f.write(json.dumps(r, default=str) + "\n")

print(f"Saved {len(batch_results)} results (with full trajectories) to {results_path}")

Saved 5 results (with full trajectories) to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/baseline_results_mlx_2b.jsonl


## Databricks Baseline (Unsloth)

Everything below runs on Databricks with GPU. Uses Unsloth to load the model directly
and `run_local_agent_loop()` for inference (parses `<tool_call>` tags from raw model output).

Skip this section if testing locally with MLX above.

In [ ]:
%pip install -q unsloth trl

In [ ]:
import json
import sys
from pathlib import Path

from unsloth import FastLanguageModel

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_FUNCTIONS, TOOL_SCHEMAS
from bbb.helpers__data_gen import (
    SYSTEM_PROMPT,
    TICKERS,
    FOCUS_AREAS,
    make_user_prompt,
    _responses_tool_to_chat,
)
from bbb.helpers__inference import (
    run_local_agent_loop,
    compute_reward,
)

# Derive valid tool names from the tool registry (not hardcoded)
VALID_TOOL_NAMES = set(TOOL_FUNCTIONS.keys())

In [ ]:
# Load Qwen3-4B — no fine-tuning, just the base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=8192,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print(f"Model loaded on {model.device}")

## Tool Setup

Convert tool schemas from Responses API (flat) to Chat Completions (nested) format,
which is what Qwen3's `apply_chat_template(tools=...)` expects.

In [ ]:
from bbb.tools import TOOL_SCHEMAS

# Convert flat Responses API schemas to nested Chat Completions format
TOOLS_CHAT = [_responses_tool_to_chat(t) for t in TOOL_SCHEMAS]

for tool in TOOLS_CHAT:
    fn = tool["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  {fn['name']}({', '.join(params)})")

## Single Ticker Test

Run one ticker with verbose output to see exactly what the raw model does.

In [ ]:
result = run_local_agent_loop(
    model=model,
    tokenizer=tokenizer,
    user_prompt="Research AAPL focusing on financial health and balance sheet strength.",
    system_prompt=SYSTEM_PROMPT,
    tools_chat=TOOLS_CHAT,
    tool_functions=TOOL_FUNCTIONS,
    max_iterations=8,
    enable_thinking=True,
    verbose=True,
)

print(f"\nIterations: {result['n_iterations']}")
print(f"Tool calls: {result['n_tool_calls']}")
print(f"Final output: {len(result['output_text'])} chars")

In [ ]:
# Show the full message flow
for i, msg in enumerate(result["messages"]):
    role = msg["role"].upper()
    content = msg.get("content", "") or ""
    tc = msg.get("tool_calls", [])

    if role == "SYSTEM":
        print(f"[{i}] {role}: {content[:80]}...")
    elif role == "USER":
        print(f"[{i}] {role}: {content}")
    elif role == "ASSISTANT" and tc:
        names = [t["function"]["name"] for t in tc]
        has_think = "<think>" in content
        print(f"[{i}] {role} (think={has_think}, tools={names})")
        if content:
            print(f"    content: {content[:150]}...")
    elif role == "ASSISTANT":
        print(f"[{i}] {role} (final): {content[:200]}...")
    elif role == "TOOL":
        print(f"[{i}] {role}: {len(content)} chars")
    print()

In [ ]:
# Score this trajectory
reward = compute_reward(result["messages"], VALID_TOOL_NAMES)
print("Reward components:")
for k, v in reward.items():
    print(f"  {k:>20}: {v}")

## Batch Evaluation

Run on ~20 tickers to get aggregate baseline metrics.

In [ ]:
import random

# Pick 20 diverse tickers
eval_tickers = random.sample(TICKERS, 20)
print(f"Evaluating: {eval_tickers}")

In [ ]:
baseline_results = []

for ticker in eval_tickers:
    focus = random.choice(FOCUS_AREAS)
    prompt = make_user_prompt(ticker, focus)

    print(f"\n{'='*50}")
    print(f"{ticker} — {focus}")

    try:
        res = run_local_agent_loop(
            model=model,
            tokenizer=tokenizer,
            user_prompt=prompt,
            system_prompt=SYSTEM_PROMPT,
            tools_chat=TOOLS_CHAT,
            tool_functions=TOOL_FUNCTIONS,
            max_iterations=8,
            enable_thinking=True,
            verbose=True,
        )

        reward = compute_reward(res["messages"], VALID_TOOL_NAMES)

        baseline_results.append({
            "ticker": ticker,
            "focus": focus,
            "n_tool_calls": res["n_tool_calls"],
            "n_iterations": res["n_iterations"],
            "output_len": len(res["output_text"]),
            "reward": reward,
            "messages": res["messages"],
        })

        print(f"  → reward={reward['total']}, tool_calls={res['n_tool_calls']}, output={len(res['output_text'])} chars")

    except Exception as e:
        print(f"  → ERROR: {e}")
        baseline_results.append({
            "ticker": ticker,
            "focus": focus,
            "error": str(e),
            "reward": {"total": -3.0},
        })

print(f"\nCompleted: {len(baseline_results)}/{len(eval_tickers)}")

## Results Summary

In [ ]:
# Aggregate metrics
rewards = [r["reward"]["total"] for r in baseline_results]
tool_counts = [r.get("n_tool_calls", 0) for r in baseline_results]
output_lens = [r.get("output_len", 0) for r in baseline_results]
completions = [r["reward"].get("completion", 0) for r in baseline_results]
valid_jsons = [r["reward"].get("valid_json", 0) for r in baseline_results]

print("=== BASELINE RESULTS ===")
print(f"Reward:       avg={sum(rewards)/len(rewards):.2f}  min={min(rewards)}  max={max(rewards)}")
print(f"Tool calls:   avg={sum(tool_counts)/len(tool_counts):.1f}  min={min(tool_counts)}  max={max(tool_counts)}")
print(f"Output chars: avg={sum(output_lens)/len(output_lens):.0f}")
print(f"Completion rate: {sum(completions)/len(completions)*100:.0f}%")
print(f"Valid JSON rate:  {sum(valid_jsons)/len(valid_jsons)*100:.0f}%")

print("\n--- Per-component averages ---")
components = ["valid_json", "thinking", "tool_selection", "efficiency", "completion", "no_hallucination"]
for comp in components:
    vals = [r["reward"].get(comp, 0) for r in baseline_results]
    print(f"  {comp:>20}: {sum(vals)/len(vals):+.2f}")

In [ ]:
# Per-ticker breakdown
print(f"{'Ticker':<8} {'Reward':>7} {'Tools':>6} {'Output':>7} {'JSON':>5} {'Done':>5}")
print("-" * 45)
for r in sorted(baseline_results, key=lambda x: x["reward"]["total"]):
    rw = r["reward"]
    print(
        f"{r['ticker']:<8} {rw['total']:>+7.2f} "
        f"{r.get('n_tool_calls', 0):>6} "
        f"{r.get('output_len', 0):>7} "
        f"{'Y' if rw.get('valid_json') else 'N':>5} "
        f"{'Y' if rw.get('completion') else 'N':>5}"
    )

## Save Baseline Results

Save for comparison with SFT and RL models.

In [ ]:
# Save results (without full messages to keep file small)
output_path = PROJECT_ROOT / "data" / "bbb" / "baseline_results.jsonl"

with open(output_path, "w") as f:
    for r in baseline_results:
        record = {k: v for k, v in r.items() if k != "messages"}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(baseline_results)} results to {output_path}")

# Also save a few full trajectories (failure examples for the talk)
examples_path = PROJECT_ROOT / "data" / "bbb" / "baseline_examples.json"
worst = sorted(baseline_results, key=lambda x: x["reward"]["total"])[:5]
best = sorted(baseline_results, key=lambda x: x["reward"]["total"], reverse=True)[:3]

with open(examples_path, "w") as f:
    json.dump({"worst": worst, "best": best}, f, indent=2, default=str)

print(f"Saved failure/success examples to {examples_path}")